# VTS Plant Catalogue (VTS_D02) — builder
Lee `data/VTS_plant_catalogue.xlsx` y genera el catálogo en PDF con el mismo diseño del manual.
Fotos: `images/<Plant ID>.jpg|png` (por ejemplo `PL-001.jpg`). Sin foto se dibuja un cuadrado.
**Cómo se usa:** `Run all`. Los textos y las plantas se editan en el Excel.

In [1]:
# Solo la primera vez: descomenta
# !pip install weasyprint openpyxl

In [2]:
import os, re, html, pathlib
from openpyxl import load_workbook
BASE_DIR = pathlib.Path(os.environ.get("VTS_DIR", ".")); DATA = BASE_DIR / "data" / "VTS_plant_catalogue.xlsx"
IMAGES = BASE_DIR / "images"; OUT = BASE_DIR / "output"; OUT.mkdir(exist_ok=True); IMAGES.mkdir(exist_ok=True)
PAGE_SIZE = "Letter"; BLUE, BLUE_L, YELLOW, INK = "#03467E", "#E6F0F9", "#F6B800", "#1F2D3D"
GREEN, RED = "#2F8F4E", "#B4361F"; SYS = {"SB":("Sediment Basins","#2B7BBF"),"TW":("Treatment Wetlands","#3C9D5B"),"VD":("Vegetated Ditches","#C99700"),"WB":("Woodchip Bioreactors","#DD6B2A")}
esc = html.escape; S = lambda v: "" if v is None else str(v).strip()

In [3]:
wb = load_workbook(DATA, data_only=True)
def rows(n):
    it = wb[n].iter_rows(values_only=True); h = next(it); return [dict(zip(h, r)) for r in it if any(c not in (None, "") for c in r)]
SET = {S(r["Key"]): S(r["Value"]) for r in rows("Settings")}
sp = lambda v: [x.strip() for x in S(v).split(";") if x.strip()]
PLANTS = [dict(id=S(r["Plant ID"]), status=S(r["Status"]), group=S(r["Group"]), sci=S(r["Scientific name"]), com=S(r["Common name"]), type=S(r["Type"]), sys=sp(r["Found in"]), ben=S(r["Benefits"]), note=S(r["Notes"]), credit=S(r["Photo credit"]), photo=S(r["Photo"]),
              form=S(r["Form"]), leaf=S(r["Leaf"]), stem=S(r["Stem"]), flower=S(r["Flower"]), clue=S(r["Clue"]), zones=sp(r["Zones"]), useful=sp(r["Useful for"]), look=S(r["Look-alike caution"]), srcs=sp(r["Sources"])) for r in rows("Plants")]
SRC_SHORT = {S(r["Key"]): S(r["Short"]) for r in rows("Sources")}
ICON_NAMES = {r["Code"]: r["Meaning"] for r in rows("Icons")}
WATCH = [dict(com=S(r["Common name"]), sci=S(r["Scientific name"]), look=S(r["Look for"]), src=S(r["Source"])) for r in rows("Watchlist")]; REFS = [S(r["Reference"]) for r in rows("References")]
NAT = [p for p in PLANTS if p["status"] == "Native"]; INV = [p for p in PLANTS if p["status"] == "Invasive"]
print(len(NAT), "nativas |", len(INV), "invasoras con ficha |", len(WATCH), "en lista de vigilancia")
missing = [p["id"] for p in PLANTS if not any((IMAGES / f'{p["photo"] or p["id"]}{e}').exists() for e in (".jpg", ".jpeg", ".png", ".webp"))]
print(f"{len(missing)} de {len(PLANTS)} plantas sin foto (cuadrado marcador)")

53 nativas | 4 invasoras con ficha | 29 en lista de vigilancia
57 de 57 plantas sin foto (cuadrado marcador)


In [4]:
FONT = (BASE_DIR / "fonts").resolve().as_uri()
CSS = f"""
@font-face {{ font-family:'Public Sans'; src:url('{FONT}/PublicSans-VF.ttf'); font-weight:100 900; }}
@font-face {{ font-family:'Public Sans'; src:url('{FONT}/PublicSans-Italic-VF.ttf'); font-weight:100 900; font-style:italic; }}
@font-face {{ font-family:'Barlow Condensed'; src:url('{FONT}/BarlowCondensed-SemiBold.ttf'); font-weight:600; }}
@font-face {{ font-family:'Barlow Condensed'; src:url('{FONT}/BarlowCondensed-Bold.ttf'); font-weight:700; }}
:root {{ --blue:{BLUE}; --blue-l:{BLUE_L}; --yellow:{YELLOW}; --ink:{INK}; }}
@page {{ size:{PAGE_SIZE}; margin:0.65in 0.6in 0.75in; @bottom-right {{ content:counter(page); font:700 10pt 'Barlow Condensed'; color:var(--blue); }} @bottom-left {{ content:'VTS Plant Catalogue · CMSF / UC Santa Cruz'; font:7.5pt 'Public Sans'; color:#6b7c8f; }} }}
@page cover {{ margin:0; @bottom-right {{content:none}} @bottom-left {{content:none}} }}
html {{ font:9.2pt/1.42 'Public Sans', sans-serif; color:var(--ink); }}
h1,h2,h3 {{ font-family:'Barlow Condensed'; font-weight:700; color:var(--blue); margin:0; }}
h1 {{ font-size:22pt; text-transform:uppercase; border-bottom:2.2pt solid var(--blue); padding-bottom:2pt; margin-bottom:2pt; line-height:1.05; }}
h2 {{ font-size:14pt; margin:16pt 0 5pt; }} p {{ margin:0 0 6pt; }} .sub {{ color:#566a7f; font-size:9pt; margin-bottom:6pt; }}
.cover {{ page:cover; height:11in; position:relative; }} .cover .photo {{ height:5.9in; }} .cover .photo .ph {{ width:100%; height:100%; border-radius:0; border:none; }}
.cover .t {{ padding:.35in .7in 0; }} .cover h1.big {{ font-size:44pt; border:none; line-height:.98; margin:0 0 8pt; }} .cover .s {{ font:700 18pt 'Barlow Condensed'; color:var(--blue); }} .cover .l {{ font-size:12pt; color:#33485d; margin-top:4pt; }} .cover .v {{ margin-top:12pt; font:600 11pt 'Barlow Condensed'; color:#566a7f; }}
.ph {{ border:1.3pt dashed #8fa9c2; border-radius:4pt; background:repeating-linear-gradient(135deg,#f1f5f9,#f1f5f9 7pt,#e6edf4 7pt,#e6edf4 14pt); color:#5b7189; display:flex; align-items:center; justify-content:center; text-align:center; font:700 7.5pt/1.3 'Barlow Condensed'; letter-spacing:.8pt; }}
.pill {{ display:inline-block; font:700 9pt 'Barlow Condensed'; letter-spacing:.8pt; color:#fff; padding:1pt 9pt; border-radius:9pt; }} .keyrow {{ display:table; width:100%; margin:4pt 0; }} .keyrow > div {{ display:table-cell; vertical-align:middle; }} .keyrow .p {{ width:1.1in; }}
.intro {{ background:var(--blue-l); border-radius:6pt; padding:7pt 10pt; margin:8pt 0 12pt; }} ul {{ margin:2pt 0 8pt; padding-left:14pt; }} li {{ margin-bottom:4pt; }}
.chip {{ display:inline-block; font:700 7.5pt 'Barlow Condensed'; letter-spacing:.6pt; color:#fff; padding:0 6pt; border-radius:8pt; margin-right:3pt; }}
.sect {{ break-before:page; }} .sect:first-of-type {{ break-before:auto; }}
table.cards {{ width:100%; border-collapse:separate; border-spacing:10pt 0; margin:0 -10pt; table-layout:fixed; }} table.cards tr {{ page-break-inside:avoid; }} table.cards > tbody > tr > td {{ vertical-align:top; padding:0 0 8pt; border:none; width:50%; background:#fff!important; }}
.card {{ border:1pt solid #c4d3e2; border-radius:7pt; overflow:hidden; height:4.0in; }} .card .bar {{ height:5pt; }} .card .ph {{ height:0.88in; margin:7pt 8pt 5pt; }} .card img {{ display:block; width:calc(100% - 16pt); height:0.88in; object-fit:cover; margin:7pt 8pt 5pt; border-radius:4pt; }}
.card .body {{ padding:0 9pt; }} .tagrow {{ margin-bottom:2pt; }} .tag {{ font:700 8pt 'Barlow Condensed'; letter-spacing:.7pt; color:#fff; padding:0 7pt; border-radius:7pt; margin-right:4pt; }} .type {{ font:600 8pt 'Barlow Condensed'; letter-spacing:.6pt; text-transform:uppercase; color:#566a7f; }}
.sci {{ font-style:italic; font-weight:700; font-size:10.5pt; color:var(--blue); line-height:1.2; }} .com {{ font-size:9pt; margin-bottom:3pt; }} .found {{ font-size:7.8pt; color:#566a7f; margin-bottom:3pt; }} .ben {{ font-size:8pt; line-height:1.32; }} .ben b {{ color:var(--blue); }}
table.clues {{ width:100%; border-collapse:collapse; margin:3pt 0 4pt; background:#f4f8fb; border-radius:4pt; }} table.clues td {{ vertical-align:middle; padding:3pt 4pt; border:none; }} table.clues td.ic {{ width:15%; text-align:center; }} .il {{ font:700 6.4pt 'Barlow Condensed'; letter-spacing:.6pt; text-transform:uppercase; color:#8aa0b5; }} .in {{ font-size:6.6pt; line-height:1.1; color:var(--ink); }} table.clues td.ct {{ font-size:7.3pt; line-height:1.28; padding-left:6pt; }}
table.usetab {{ width:100%; border-collapse:collapse; font-size:7.4pt; margin:2pt 0 2pt; }} table.usetab th {{ background:var(--blue); color:#fff; padding:1.5pt 4pt; text-align:left; font-weight:600; }} table.usetab td {{ padding:1.5pt 4pt; border-bottom:.5pt solid #d3deea; }} table.usetab td:nth-child(n+2) {{ text-align:center; color:{GREEN}; font-weight:700; }}
.zone {{ font-size:7.6pt; margin-bottom:2pt; }} .zone b {{ color:var(--blue); }} .ufs {{ margin:2pt 0 3pt; }} .uf {{ display:inline-block; font:600 6.8pt 'Barlow Condensed'; letter-spacing:.4pt; background:#e3f2e8; color:#1f6b3a; padding:0 5pt; border-radius:6pt; margin:0 3pt 2pt 0; }}
.use {{ font-size:7.6pt; padding:2pt 5pt; border-radius:3pt; margin:2pt 0; }} .use.rm {{ background:#fbe3dd; color:#7a1f10; }} .use.chk {{ background:#fff4cf; color:#5a4300; }}
table.zonebar {{ width:100%; border-collapse:separate; border-spacing:2pt 0; margin:2pt 0 2pt; table-layout:fixed; }} table.zonebar td {{ font:600 6.8pt 'Barlow Condensed'; letter-spacing:.5pt; text-transform:uppercase; text-align:center; padding:1.5pt 0; border-radius:3pt; background:#eef2f6; color:#9aaabb; border:none; }} table.zonebar td.on {{ background:#3C9D5B; color:#fff; }} table.zonebar td.zl {{ width:24%; background:none; color:var(--blue); text-align:left; font-size:7pt; }}
.look {{ font-size:7.3pt; line-height:1.28; margin-top:2pt; background:#fff4cf; color:#5a4300; padding:2pt 5pt; border-radius:3pt; }} .src {{ font-size:6.4pt; color:#8aa0b5; margin-top:2pt; }} h2.grp {{ font-size:13pt; margin:8pt 0 4pt; page-break-after:avoid; }}
.note {{ margin-top:3pt; background:#fbe3dd; color:#7a1f10; font-size:7.6pt; padding:2pt 5pt; border-radius:3pt; line-height:1.28; }}
table.watch {{ border-collapse:collapse; width:100%; font-size:8.6pt; }} table.watch th {{ background:{RED}; color:#fff; text-align:left; padding:3pt 6pt; }} table.watch td {{ padding:2.5pt 6pt; font-size:8.2pt; border-bottom:.6pt solid #e7b9b0; }} table.watch td.s {{ font-style:italic; }}
.refs p {{ padding-left:12pt; text-indent:-12pt; font-size:8.4pt; word-break:break-word; }}
"""
G_, B_, Y_, W_ = "#3C9D5B", "#8a6a3b", "#F6B800", "#2B7BBF"
ICONS = {
 "form_T": f'<rect x="10.8" y="12" width="2.6" height="10" fill="{B_}"/><circle cx="12" cy="9" r="7" fill="{G_}"/>',
 "form_S": f'<rect x="11.2" y="16" width="1.6" height="6" fill="{B_}"/><circle cx="7.5" cy="14" r="4.8" fill="{G_}"/><circle cx="16.5" cy="14" r="4.8" fill="#2F8F4E"/><circle cx="12" cy="9" r="4.8" fill="{G_}"/>',
 "form_G": f'<g stroke="{G_}" stroke-width="2" fill="none" stroke-linecap="round"><path d="M12 22 C11 14 7 9 4 5"/><path d="M12 22 V3"/><path d="M12 22 C13 14 17 9 20 5"/></g>',
 "form_R": f'<g stroke="{G_}" stroke-width="2" stroke-linecap="round"><path d="M7 22 V7"/><path d="M12 22 V4"/><path d="M17 22 V8"/></g><ellipse cx="12" cy="4" rx="1.8" ry="2.6" fill="{B_}"/><ellipse cx="7" cy="7" rx="1.5" ry="2.2" fill="{B_}"/>',
 "form_F": f'<path d="M12 22 V10" stroke="{G_}" stroke-width="2"/><circle cx="12" cy="7" r="4.6" fill="{Y_}"/><circle cx="12" cy="7" r="1.8" fill="{B_}"/><path d="M12 18 C7 18 5 15 5 13 C9 13 12 14 12 18Z" fill="{G_}"/>',
 "form_AF": f'<path d="M2 16 q3 -3 6 0 t6 0 t6 0" stroke="{W_}" stroke-width="2" fill="none"/><circle cx="7" cy="12.5" r="2.4" fill="{G_}"/><circle cx="12.5" cy="11" r="2.4" fill="{G_}"/><circle cx="17.5" cy="12.5" r="2.4" fill="{G_}"/>',
 "form_AS": f'<path d="M2 5 q3 -3 6 0 t6 0 t6 0" stroke="{W_}" stroke-width="2" fill="none"/><path d="M12 22 V9 M12 12 l-5 -2 M12 12 l5 -2 M12 16 l-5 -2 M12 16 l5 -2" stroke="{G_}" stroke-width="1.7" fill="none"/>',
 "form_M": f'<g stroke="{G_}" stroke-width="2" fill="none" stroke-linecap="round"><path d="M4 21 C6 13 9 13 8 6"/><path d="M10 22 C12 14 15 13 13 5"/><path d="M17 21 C19 14 20 12 20 7"/></g>',
 "form_C": f'<g stroke="{B_}" stroke-width="2"><path d="M9 22 V5"/><path d="M15 22 V7"/><path d="M9 11 H15 M9 16 H15" stroke-width="1.2"/></g><path d="M5 5 q4 -4 7 0 q4 -4 7 2 q-7 3 -14 -2Z" fill="#c9b08a"/>',
 "form_V": f'<path d="M3 21 C8 15 8 10 12 9 C16 8 16 5 21 4" stroke="{G_}" stroke-width="2" fill="none"/><path d="M8 15 l-5 -1 l4 -4Z M14 8 l4 3 l1 -5Z" fill="#2F8F4E"/>',
 "leaf_BL": f'<path d="M3 21 C6 10 13 5 21 3 C18 11 11 17 3 21Z" fill="{G_}"/>',
 "leaf_RS": f'<rect x="9.5" y="2" width="5" height="20" rx="2.5" fill="{G_}"/>',
 "leaf_LA": f'<path d="M12 2 C19 8 19 16 12 22 C5 16 5 8 12 2Z" fill="{G_}"/><path d="M12 5 V19" stroke="#fff" stroke-width="1.2"/>',
 "leaf_OV": f'<ellipse cx="12" cy="12" rx="8" ry="10" fill="{G_}"/><path d="M12 3 V21" stroke="#fff" stroke-width="1.2"/>',
 "leaf_LO": f'<path d="M12 2 l3 4 l5 -1 l-2 5 l4 3 l-5 2 l1 5 l-5 -2 l-3 3 l-3 -3 l-5 2 l1 -5 l-5 -2 l4 -3 l-2 -5 l5 1Z" fill="{G_}"/>',
 "leaf_CP": f'<path d="M12 3 V21" stroke="#2F8F4E" stroke-width="1.5"/><g fill="{G_}"><ellipse cx="7.5" cy="6.5" rx="3.6" ry="1.8"/><ellipse cx="16.5" cy="6.5" rx="3.6" ry="1.8"/><ellipse cx="7" cy="12" rx="3.8" ry="1.8"/><ellipse cx="17" cy="12" rx="3.8" ry="1.8"/><ellipse cx="7.5" cy="17.5" rx="3.4" ry="1.7"/><ellipse cx="16.5" cy="17.5" rx="3.4" ry="1.7"/></g>',
 "leaf_FE": f'<path d="M12 3 V21 M12 7 l-7 -3 M12 7 l7 -3 M12 12 l-8 -3 M12 12 l8 -3 M12 17 l-7 -3 M12 17 l7 -3" stroke="{G_}" stroke-width="1.7" fill="none" stroke-linecap="round"/>',
 "leaf_TI": f'<circle cx="7" cy="12" r="3.4" fill="{G_}"/><circle cx="16" cy="8" r="3.4" fill="{G_}"/><circle cx="15" cy="16.5" r="3.4" fill="{G_}"/>',
 "stem_WO": f'<rect x="8" y="2" width="8" height="20" fill="{B_}"/><path d="M10.5 4 V20 M13.5 5 V19" stroke="#5e4424" stroke-width="1"/>',
 "stem_HC": f'<circle cx="12" cy="12" r="9" fill="#c9b08a"/><circle cx="12" cy="12" r="5" fill="#fff"/>',
 "stem_TR": f'<path d="M12 3 L21.5 20 H2.5Z" fill="{G_}"/>',
 "stem_RO": f'<circle cx="12" cy="12" r="9" fill="{G_}"/>',
 "stem_HB": f'<path d="M12 3 V21" stroke="{G_}" stroke-width="3.4"/><path d="M12 9 l7 -3 M12 15 l-7 -3" stroke="{G_}" stroke-width="2.2"/>',
 "stem_FL": f'<path d="M2 14 q3 -3 6 0 t6 0 t6 0" stroke="{W_}" stroke-width="2.4" fill="none"/>'}
def icon(group, code):
    svg = ICONS.get(f"{group}_{code}", "")
    return f'<svg viewBox="0 0 24 24" width="26" height="26">{svg}</svg>' if svg else ""
FLOWER_HEX = {"white": "#f3f3ee", "yellow": "#F6B800", "orange": "#F28C28", "red": "#D6342C", "pink": "#E87AA6", "magenta": "#C2308A", "blue-violet": "#6B5BD2", "blue-purple": "#7A4FC0", "cream": "#F1E6C3", "red-orange": "#E4572E", "red-yellow": "#E0702B"}
def flower_icon(color):
    h = FLOWER_HEX.get(color.lower(), "#F6B800"); petals = "".join(f'<ellipse cx="12" cy="5.6" rx="3" ry="4.2" fill="{h}" stroke="#00000022" stroke-width=".6" transform="rotate({a} 12 12)"/>' for a in (0, 72, 144, 216, 288))
    return f'<svg viewBox="0 0 24 24" width="26" height="26">{petals}<circle cx="12" cy="12" r="2.6" fill="#8a6a3b"/></svg>'
def clues(p):
    cells = ""
    for g, lab, code in (("form", "Form", p["form"]), ("leaf", "Leaf", p["leaf"]), ("stem", "Stem", p["stem"])):
        if code: cells += f'<td class="ic">{icon(g, code)}<div class="il">{lab}</div><div class="in">{esc(ICON_NAMES.get(code, code))}</div></td>'
    if p["flower"]: cells += f'<td class="ic">{flower_icon(p["flower"])}<div class="il">Flower</div><div class="in">{esc(p["flower"])}</div></td>'
    return f'<table class="clues"><tr>{cells}<td class="ct">{esc(p["clue"])}</td></tr></table>' if cells else ""
ZN = [("W", "Water"), ("L", "Low"), ("M", "Mid"), ("H", "High")]
def zonebar(p):
    if not p["zones"]: return ""
    segs = "".join(f'<td class="z{" on" if c in p["zones"] else ""}">{n}</td>' for c, n in ZN)
    return f'<table class="zonebar"><tr><td class="zl">Grows best in</td>{segs}</tr></table>'
def usebox(p):
    chips = "".join(f'<span class="uf">{esc(u)}</span>' for u in p["useful"])
    if p["status"] == "Invasive": return '<div class="use rm"><b>Remove and report</b> to the supervisor.</div>'
    return f'{zonebar(p)}<div class="ufs">{chips}</div>'
def photo(p, h="1.18in"):
    for e in (".jpg", ".jpeg", ".png", ".webp"):
        f = IMAGES / f'{p["photo"] or p["id"]}{e}'
        if f.exists(): return f'<img src="{f.resolve().as_uri()}">'
    return f'<div class="ph"><span>INSERT PHOTO<br>{esc(p["id"])}</span></div>'
def card(p):
    col = GREEN if p["status"] == "Native" else RED
    chips = "".join(f'<span class="chip" style="background:{SYS[s][1]}">{s}</span>' for s in p["sys"] if s in SYS)
    note = f'<div class="note">{esc(p["note"])}</div>' if p["note"] else ""
    look = f'<div class="look"><b>Do not confuse:</b> {esc(p["look"])}</div>' if p["look"] else ""
    return (f'<div class="card"><div class="bar" style="background:{col}"></div>{photo(p)}<div class="body"><div class="tagrow"><span class="tag" style="background:{col}">{p["status"].upper()}</span><span class="type">{esc(p["type"])}</span></div>'
            f'<div class="sci">{esc(p["sci"])}</div><div class="com">{esc(p["com"])}</div>{clues(p)}<div class="found">Found in: {chips}</div>{usebox(p)}<div class="ben"><b>Benefits</b> {esc(p["ben"])}</div>{look}{note}<div class="src">Source: {esc("; ".join(SRC_SHORT.get(x, x) for x in p["srcs"]))}{(" · Photo: " + esc(p["credit"])) if p["credit"] else ""}</div></div></div>')
def grid(ps):
    out = ""
    for i in range(0, len(ps), 2):
        pair = ps[i:i + 2]; out += "<tr>" + "".join(f"<td>{card(p)}</td>" for p in pair) + ("<td></td>" if len(pair) == 1 else "") + "</tr>"
    return f'<table class="cards"><tbody>{out}</tbody></table>'
def how(t):
    parts = t.split("\n"); out = f"<p>{esc(parts[0])}</p><ul>" + "".join(f"<li>{esc(x[2:])}</li>" for x in parts[1:] if x.startswith("- ")) + "</ul>"; return out
cover = (f'<section class="cover"><div class="photo"><div class="ph"><span>COVER PHOTO</span></div></div><div class="t"><h1 class="big">{esc(SET["Title"]).replace("CATALOGUE","<br>CATALOGUE")}</h1><div class="s">{esc(SET["Subtitle"])}</div>'
         f'<div class="l">{esc(SET["Lede"])}</div><div class="v">{esc(SET["Companion"])}<br>{esc(SET["Version line"])}</div></div></section>')
key = (f'<section class="sect"><h1>Getting started</h1><h2>Color key</h2><div class="keyrow"><div class="p"><span class="pill" style="background:{GREEN}">NATIVE</span></div><div>{esc(SET["Native key"])}</div></div>'
       f'<div class="keyrow"><div class="p"><span class="pill" style="background:{RED}">INVASIVE</span></div><div>{esc(SET["Invasive key"])}</div></div><h2>Abbreviations</h2>'
       + "".join(f'<div class="keyrow"><div class="p"><span class="chip" style="background:{c}">{k}</span></div><div>{n}</div></div>' for k, (n, c) in SYS.items())
       + f'<h2>How to use this catalogue</h2><div class="intro">{how(SET["How to use"])}</div><p class="sub">{esc(SET["Zone note"])}</p></section>')
GROUPS = []
for p in NAT:
    if p["group"] not in GROUPS: GROUPS.append(p["group"])
nat = f'<section class="sect"><h1>{esc(SET["Native heading"])}</h1><div class="sub">{esc(SET["Native subheading"])}</div>' + "".join(f'<h2 class="grp">{esc(g)}</h2>{grid([p for p in NAT if p["group"] == g])}' for g in GROUPS) + "</section>"
watch = ('<h2>Other invasive plants to watch for</h2><p>Remove and report to the supervisor. Use gloves; some plants are toxic.</p><table class="watch"><thead><tr><th>Common name</th><th>Scientific name</th><th>What to look for</th></tr></thead><tbody>'
         + "".join(f'<tr><td>{esc(w["com"])}{" †" if w["src"] == "Proposed" else ""}</td><td class="s">{esc(w["sci"])}</td><td>{esc(w["look"])}</td></tr>' for w in WATCH) + f'</tbody></table><p class="sub">{esc(SET["Watch note"])}</p>')
inv = f'<section class="sect"><h1>{esc(SET["Invasive heading"])}</h1><div class="sub">{esc(SET["Invasive subheading"])}</div>{grid(INV)}{watch}</section>'
refs = '<section class="sect"><h1>References</h1><div class="refs">' + "".join(f"<p>{esc(r)}</p>" for r in REFS) + "</div></section>"
HTML = f'<!doctype html><html lang="en"><head><meta charset="utf-8"><title>{esc(SET["Title"])}</title><style>{CSS}</style></head><body>{cover}{key}{nat}{inv}{refs}</body></html>'

In [5]:
from weasyprint import HTML as WP
OUT_PDF = OUT / "VTS_D02_Plant_Catalogue.pdf"; doc = WP(string=HTML, base_url=str(BASE_DIR)).render(); doc.write_pdf(str(OUT_PDF)); print(f"PDF: {len(doc.pages)} páginas → {OUT_PDF}")

PDF: 20 páginas → output\VTS_D02_Plant_Catalogue.pdf
